In [ ]:
# ---------------------------------------------------------------------------
# STEP 2: Feature Engineering & Preprocessing with RDKit
# ---------------------------------------------------------------------------

# 1. Install RDKit and dependencies
!pip install -q rdkit pandas numpy matplotlib

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
import os

# 2. Load the dataset saved from Step 1
# Replace filename if your ChEMBL ID differed
input_file = 'CHEMBL203_IC50_bioactivity_data.csv'

if not os.path.exists(input_file):
    raise FileNotFoundError(f"Cannot find {input_file}. Ensure Step 1 script ran successfully!")

df = pd.read_csv(input_file)
print(f"Loaded {len(df)} rows from {input_file}.")


# 3. Function to convert IC50 (nM) to pIC50 (-log10(M))
def convert_to_pic50(input_df):
    pIC50 = []

    for val in input_df['standard_value']:
        # Cap high values to 100,000,000 nM to prevent negative pIC50 values from noise
        if val > 100000000:
            val = 100000000
        # Prevent log(0) errors
        if val <= 0:
            val = 1e-9

        molar = val * 1e-9  # Convert nM to Molar
        pIC50.append(-np.log10(molar))

    input_df['pIC50'] = pIC50
    return input_df

print("\nConverting IC50 to pIC50...")
df = convert_to_pic50(df)


# 4. Function to calculate Lipinski Descriptors (Molecular Weight, LogP, NumHDonors, NumHAcceptors)
def calculate_lipinski(smiles_list):
    mol_data = []

    for sm in smiles_list:
        mol = Chem.MolFromSmiles(sm)
        if mol is not None:
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            hdonors = Descriptors.NumHDonors(mol)
            hacceptors = Descriptors.NumHAcceptors(mol)
            mol_data.append([mw, logp, hdonors, hacceptors])
        else:
            mol_data.append([np.nan, np.nan, np.nan, np.nan])

    descriptors_df = pd.DataFrame(
        mol_data,
        columns=['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']
    )
    return descriptors_df

print("Calculating Lipinski Descriptors...")
lipinski_df = calculate_lipinski(df['canonical_smiles'])
df = pd.concat([df, lipinski_df], axis=1)


# 5. Function to generate Morgan Fingerprints (ECFP4 - Radius 2, 1024 bits)
def generate_morgan_fingerprints(smiles_list, radius=2, n_bits=1024):
    fingerprints = []

    for sm in smiles_list:
        mol = Chem.MolFromSmiles(sm)
        if mol is not None:
            # Generate bit vector for Morgan Fingerprint (ECFP4 equivalent)
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            # Convert bit vector to numpy array
            arr = np.zeros((0,), dtype=int)
            Chem.DataStructs.ConvertToNumpyArray(fp, arr)
            fingerprints.append(arr)
        else:
            fingerprints.append(np.zeros((n_bits,), dtype=int))

    # Create DataFrame where each column represents 1 bit feature (Bit_0 to Bit_1023)
    bit_columns = [f"Bit_{i}" for i in range(n_bits)]
    fp_df = pd.DataFrame(fingerprints, columns=bit_columns)
    return fp_df

print("Generating 1024-bit Morgan Fingerprints (ECFP4)...")
fp_df = generate_morgan_fingerprints(df['canonical_smiles'])


# 6. Combine all features into a final dataset ready for Machine Learning
final_df = pd.concat([df, fp_df], axis=1)

# Drop any rows where SMILES parsing failed
final_df = final_df.dropna(subset=['MW', 'pIC50'])

# Save processed dataset
output_file = 'processed_qsar_dataset.csv'
final_df.to_csv(output_file, index=False)

print(f"\n--- SUCCESS ---")
print(f"Final Dataset Shape: {final_df.shape} (Rows, Columns)")
print(f"Features Generated: 4 Lipinski Descriptors + 1024 Fingerprint Bits")
print(f"Target Variable: pIC50")
print(f"Dataset saved to: {output_file}")

# Preview target distribution
print("\npIC50 Summary Stats:")
print(final_df['pIC50'].describe())